## Cell 1 — Import library

In [1]:
import pandas as pd
from pathlib import Path
import pm4py

## Cell 2 — Tentukan folder project

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables


## Cell 3 — Baca compact log

In [3]:
COMPACT_LOG_PATH = PROCESSED_DIR / "04_compact_log_process_ready.csv"

compact_log = pd.read_csv(
    COMPACT_LOG_PATH,
    parse_dates=["timestamp"]
)

print("Ukuran compact_log:", compact_log.shape)
display(compact_log.head())

print("\nJumlah mahasiswa:", compact_log["case_id"].nunique())
print("Jumlah activity_mapped unik:", compact_log["activity_mapped"].nunique())

Ukuran compact_log: (116611, 18)


,case_id,activity,timestamp,kelas,Component,Event context,Description,Origin,IP address,nim,nama,nilai_total,nilai_indeks,label_performance,activity_mapped,activity_category,event_order,compact_event_order
0,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 13:30:09,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.204,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,1,1
1,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 14:24:42,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Material Viewed,material,4,2
2,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 15:41:24,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,5,3
3,ADITYA PUTRA PERMANA,Course module viewed,2025-09-18 15:41:57,IF-49-02,File,File: Materi 1 Sistem Bilangan Riil dan Pertak...,The user with id &#039;124462&#039; viewed the...,web,103.233.100.202,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Material Viewed,material,6,4
4,ADITYA PUTRA PERMANA,Course viewed,2025-09-18 16:40:55,IF-49-02,System,Course: KALKULUS IF-49-02 [DLW],The user with id &#039;124462&#039; viewed the...,web,36.72.198.4,103012530065,ADITYA PUTRA PERMANA,73.01,B,Sedang,Course Viewed,course,7,5



Jumlah mahasiswa: 89
Jumlah activity_mapped unik: 25


## Cell 4 — Buat PM4Py dataframe

In [4]:
pm_df = compact_log[[
    "case_id",
    "activity_mapped",
    "timestamp"
]].copy()

pm_df = pm_df.rename(columns={
    "case_id": "case:concept:name",
    "activity_mapped": "concept:name",
    "timestamp": "time:timestamp"
})

pm_df = pm_df.sort_values(
    by=["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

pm_df = pm4py.format_dataframe(
    pm_df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp"
)

print("PM dataframe siap")
print("Jumlah case:", pm_df["case:concept:name"].nunique())
print("Jumlah event:", len(pm_df))
print("Jumlah activity:", pm_df["concept:name"].nunique())

PM dataframe siap
Jumlah case: 89
Jumlah event: 116611
Jumlah activity: 25


## Cell 5 — Buat main learning dataframe

In [5]:
main_learning_activities = [
    "Course Viewed",
    "Material Viewed",
    "Quiz Module Viewed",
    "Quiz Access Prevented",
    "Quiz Started",
    "Quiz Viewed",
    "Quiz Updated",
    "Quiz Auto-saved",
    "Quiz Summary Viewed",
    "Quiz Submitted"
]

main_learning_df = pm_df[
    pm_df["concept:name"].isin(main_learning_activities)
].copy()

main_learning_df = main_learning_df.sort_values(
    by=["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

print("Ukuran main_learning_df:", main_learning_df.shape)
print("Jumlah case:", main_learning_df["case:concept:name"].nunique())
print("Jumlah activity:", main_learning_df["concept:name"].nunique())

display(main_learning_df["concept:name"].value_counts())

Ukuran main_learning_df: (113803, 5)
Jumlah case: 89
Jumlah activity: 10


concept:name
Quiz Viewed              42249
Quiz Updated             41209
Quiz Module Viewed        7348
Course Viewed             6384
Quiz Auto-saved           4594
Material Viewed           3405
Quiz Summary Viewed       2362
Quiz Started              2259
Quiz Submitted            2210
Quiz Access Prevented     1783
Name: count, dtype: Int64

## Cell 6 — Convert ke event log PM4Py

In [6]:
main_learning_event_log = pm4py.convert_to_event_log(main_learning_df)

print("Main learning event log berhasil dibuat")
print("Jumlah trace/case:", len(main_learning_event_log))

Main learning event log berhasil dibuat
Jumlah trace/case: 89


## Cell 7 — Discovery ulang Petri net main learning

In [7]:
main_net, main_initial_marking, main_final_marking = pm4py.discover_petri_net_inductive(
    main_learning_event_log
)

print("Main learning Petri net berhasil ditemukan.")
print("Jumlah places:", len(main_net.places))
print("Jumlah transitions:", len(main_net.transitions))
print("Initial marking:", main_initial_marking)
print("Final marking:", main_final_marking)

Main learning Petri net berhasil ditemukan.
Jumlah places: 28
Jumlah transitions: 31
Initial marking: ['source:1']
Final marking: ['sink:1']


## Cell 8 — Hitung fitness

In [8]:
fitness_main = pm4py.fitness_token_based_replay(
    main_learning_event_log,
    main_net,
    main_initial_marking,
    main_final_marking
)

print("Fitness result:")
print(fitness_main)

replaying log with TBR, completed traces ::   0%|          | 0/89 [00:00<?, ?it/s]

Fitness result:
{'perc_fit_traces': 100.0, 'average_trace_fitness': 1.0, 'log_fitness': 1.0, 'percentage_of_fitting_traces': 100.0}


## Cell 9 — Hitung precision

In [9]:
precision_main = pm4py.precision_token_based_replay(
    main_learning_event_log,
    main_net,
    main_initial_marking,
    main_final_marking
)

print("Precision result:")
print(precision_main)

replaying log with TBR, completed traces ::   0%|          | 0/113156 [00:00<?, ?it/s]

Precision result:
0.10105269632610736


## Cell 10 — Hitung generalization dan simplicity

In [10]:
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator

generalization_main = generalization_evaluator.apply(
    main_learning_event_log,
    main_net,
    main_initial_marking,
    main_final_marking
)

simplicity_main = simplicity_evaluator.apply(main_net)

print("Generalization:", generalization_main)
print("Simplicity:", simplicity_main)

replaying log with TBR, completed traces ::   0%|          | 0/89 [00:00<?, ?it/s]

Generalization: 0.8658424861096414
Simplicity: 0.5841584158415842


## Cell 11 — Rapikan hasil conformance ke tabel

In [11]:
def extract_fitness_value(fitness_result):
    if isinstance(fitness_result, dict):
        if "log_fitness" in fitness_result:
            return fitness_result["log_fitness"]
        elif "average_trace_fitness" in fitness_result:
            return fitness_result["average_trace_fitness"]
    return fitness_result


def extract_fit_trace_percentage(fitness_result):
    if isinstance(fitness_result, dict):
        if "perc_fit_traces" in fitness_result:
            return fitness_result["perc_fit_traces"]
        elif "percentage_of_fitting_traces" in fitness_result:
            return fitness_result["percentage_of_fitting_traces"]
    return None


conformance_summary_main = pd.DataFrame({
    "model": ["Main Learning Log - Inductive Miner"],
    "jumlah_case": [main_learning_df["case:concept:name"].nunique()],
    "jumlah_event": [len(main_learning_df)],
    "jumlah_activity": [main_learning_df["concept:name"].nunique()],
    "fitness": [extract_fitness_value(fitness_main)],
    "fit_trace_percentage": [extract_fit_trace_percentage(fitness_main)],
    "precision": [precision_main],
    "generalization": [generalization_main],
    "simplicity": [simplicity_main],
    "jumlah_places": [len(main_net.places)],
    "jumlah_transitions": [len(main_net.transitions)]
})

display(conformance_summary_main)

,model,jumlah_case,jumlah_event,jumlah_activity,fitness,fit_trace_percentage,precision,generalization,simplicity,jumlah_places,jumlah_transitions
0,Main Learning Log - Inductive Miner,89,113803,10,1.0,100.0,0.101053,0.865842,0.584158,28,31


## Cell 12 — Simpan ringkasan conformance

In [12]:
CONFORMANCE_MAIN_PATH = OUTPUT_TABLES_DIR / "07_conformance_main_learning.csv"

conformance_summary_main.to_csv(
    CONFORMANCE_MAIN_PATH,
    index=False
)

print("Ringkasan conformance main learning berhasil disimpan ke:")
print(CONFORMANCE_MAIN_PATH)

Ringkasan conformance main learning berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\07_conformance_main_learning.csv


## Cell 13 — Ambil diagnostic token replay per trace

In [13]:
tbr_diagnostics = pm4py.conformance_diagnostics_token_based_replay(
    main_learning_event_log,
    main_net,
    main_initial_marking,
    main_final_marking
)

print("Jumlah diagnostics:", len(tbr_diagnostics))
print("Contoh diagnostic trace pertama:")
print(tbr_diagnostics[0])

replaying log with TBR, completed traces ::   0%|          | 0/89 [00:00<?, ?it/s]

Jumlah diagnostics: 89
Contoh diagnostic trace pertama:
{'trace_is_fit': True, 'trace_fitness': 1.0, 'activated_transitions': [(tau_1, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (init_loop_44, None), (f2a80a89-ab58-46f3-9c93-cecf9145e8a1, 'Material Viewed'), (skip_28, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (skip_46, None), (f2a80a89-ab58-46f3-9c93-cecf9145e8a1, 'Material Viewed'), (skip_28, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (01040ae3-b453-41f0-a847-10e3283d14ca, 'Quiz Module Viewed'), (skip_28, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (skip_46, None), (f2a80a89-ab58-46f3-9c93-cecf9145e8a1, 'Material Viewed'), (skip_28, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (skip_46, None), (f2a80a89-ab58-46f3-9c93-cecf9145e8a1, 'Material Viewed'), (skip_28, None), (7eb99697-e858-4f79-8b33-64ac39210485, 'Course Viewed'), (skip_46, None), (f2a80a89-ab58-46f3-9c93-cecf9145e8a1, 'Mat

## Cell 14 — Buat tabel fitness per mahasiswa

In [14]:
trace_diagnostics_rows = []

for trace, diagnostic in zip(main_learning_event_log, tbr_diagnostics):
    case_id = trace.attributes.get("concept:name")
    
    trace_diagnostics_rows.append({
        "case_id": case_id,
        "trace_is_fit": diagnostic.get("trace_is_fit"),
        "trace_fitness": diagnostic.get("trace_fitness"),
        "missing_tokens": diagnostic.get("missing_tokens"),
        "consumed_tokens": diagnostic.get("consumed_tokens"),
        "remaining_tokens": diagnostic.get("remaining_tokens"),
        "produced_tokens": diagnostic.get("produced_tokens")
    })

trace_fitness_df = pd.DataFrame(trace_diagnostics_rows)

display(trace_fitness_df.head())

print("Ringkasan trace fitness:")
display(trace_fitness_df["trace_fitness"].describe())

,case_id,trace_is_fit,trace_fitness,missing_tokens,consumed_tokens,remaining_tokens,produced_tokens
0,ADITYA PUTRA PERMANA,True,1.0,0,3866,0,3866
1,ADY RAHMAN WICAKSONO,True,1.0,0,2096,0,2096
2,AGUNG KALEB SASAUW,True,1.0,0,2488,0,2488
3,ALFITO MAULANA,True,1.0,0,1856,0,1856
4,ALFONSUS LIGUORI DANGO,True,1.0,0,1684,0,1684


Ringkasan trace fitness:


count    89.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: trace_fitness, dtype: float64

## Cell 15 — Gabungkan trace fitness dengan label performa

In [15]:
LABEL_PATH = PROCESSED_DIR / "02_label_performance.csv"

label_df = pd.read_csv(LABEL_PATH, dtype={"nim": str})

# Mapping VANIA harus sama seperti preprocessing
label_df["nama_key"] = label_df["nama"].str.upper().str.split().str.join(" ")

manual_name_mapping = {
    "VANIA": "VANIA VANIA"
}

label_df["match_key"] = label_df["nama_key"].replace(manual_name_mapping)

trace_fitness_labeled = trace_fitness_df.merge(
    label_df[["match_key", "nim", "nama", "kelas", "nilai_total", "nilai_indeks", "label_performance"]],
    left_on="case_id",
    right_on="match_key",
    how="left"
).drop(columns=["match_key"])

display(trace_fitness_labeled.head())

print("Missing label:")
display(trace_fitness_labeled[["nim", "label_performance"]].isna().sum())

print("\nTrace fitness berdasarkan label performance:")
display(
    trace_fitness_labeled
    .groupby("label_performance")["trace_fitness"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

,case_id,trace_is_fit,trace_fitness,missing_tokens,consumed_tokens,remaining_tokens,produced_tokens,nim,nama,kelas,nilai_total,nilai_indeks,label_performance
0,ADITYA PUTRA PERMANA,True,1.0,0,3866,0,3866,103012530065,ADITYA PUTRA PERMANA,IF-49-02,73.01,B,Sedang
1,ADY RAHMAN WICAKSONO,True,1.0,0,2096,0,2096,103012500077,ADY RAHMAN WICAKSONO,IF-49-03,60.04,BC,Sedang
2,AGUNG KALEB SASAUW,True,1.0,0,2488,0,2488,103012500211,AGUNG KALEB SASAUW,IF-49-03,75.04,AB,Tinggi
3,ALFITO MAULANA,True,1.0,0,1856,0,1856,103012530032,ALFITO MAULANA,IF-49-03,67.24,B,Sedang
4,ALFONSUS LIGUORI DANGO,True,1.0,0,1684,0,1684,103012580046,ALFONSUS LIGUORI DANGO,IF-49-02,40.28,D,Rendah


Missing label:


nim                  0
label_performance    0
dtype: int64


Trace fitness berdasarkan label performance:


,count,mean,median,min,max
label_performance,,,,,
Rendah,15,1.0,1.0,1.0,1.0
Sedang,54,1.0,1.0,1.0,1.0
Tinggi,20,1.0,1.0,1.0,1.0


## Cell 16 — Simpan trace fitness per mahasiswa

In [16]:
TRACE_FITNESS_PATH = PROCESSED_DIR / "07_trace_fitness_main_learning.csv"
TRACE_FITNESS_TABLE_PATH = OUTPUT_TABLES_DIR / "07_trace_fitness_by_student.csv"

trace_fitness_labeled.to_csv(TRACE_FITNESS_PATH, index=False)
trace_fitness_labeled.to_csv(TRACE_FITNESS_TABLE_PATH, index=False)

print("Trace fitness per mahasiswa berhasil disimpan ke:")
print(TRACE_FITNESS_PATH)
print(TRACE_FITNESS_TABLE_PATH)

Trace fitness per mahasiswa berhasil disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed\07_trace_fitness_main_learning.csv
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\07_trace_fitness_by_student.csv


## Kesimpulan Conformance Checking

Pada tahap conformance checking, model Petri net hasil Inductive Miner pada main learning log dievaluasi menggunakan metrik fitness, precision, generalization, dan simplicity. Main learning log terdiri dari 89 trace mahasiswa, 113.803 event, dan 10 aktivitas utama yang berkaitan dengan akses course, material, dan pengerjaan kuis.

Hasil token-based replay menunjukkan nilai fitness sebesar 1,0 dengan fit trace percentage sebesar 100%. Hal ini menunjukkan bahwa model proses mampu merepresentasikan seluruh trace mahasiswa pada main learning log. Seluruh trace mahasiswa dapat direplay oleh model tanpa missing token maupun remaining token, sehingga trace fitness pada kelompok performa Rendah, Sedang, dan Tinggi sama-sama bernilai 1,0.

Namun, nilai precision sebesar 0,1011 menunjukkan bahwa model memiliki fleksibilitas yang tinggi. Dengan kata lain, meskipun model mampu menjelaskan seluruh perilaku yang muncul pada log, model juga masih mengizinkan banyak kemungkinan alur aktivitas lain yang tidak secara langsung muncul pada event log. Kondisi ini sejalan dengan karakteristik log LMS yang memiliki banyak variasi urutan dan pengulangan aktivitas, terutama pada siklus pengerjaan kuis.

Nilai generalization sebesar 0,8658 menunjukkan bahwa model memiliki kemampuan generalisasi yang cukup baik terhadap variasi trace pada log. Sementara itu, nilai simplicity sebesar 0,5842 menunjukkan bahwa model memiliki kompleksitas sedang. Kompleksitas ini disebabkan oleh adanya aktivitas yang berulang dan fleksibel, seperti Quiz Viewed, Quiz Updated, Quiz Auto-saved, Quiz Summary Viewed, dan Quiz Submitted.

Berdasarkan hasil tersebut, model proses dapat dikatakan mampu merepresentasikan pola utama proses belajar mahasiswa, tetapi masih memiliki precision yang rendah karena proses belajar pada LMS bersifat fleksibel dan tidak linear. Oleh karena itu, hasil conformance checking digunakan sebagai dasar untuk memahami kualitas model proses, sedangkan perbedaan karakteristik mahasiswa akan dianalisis lebih lanjut melalui ekstraksi fitur proses pada tahap berikutnya.